# N-Queens: transpile Prolog → ClojureScript (in-app)

Shows the concise **Prolog** n-queens and **transpiles it to ClojureScript** with UnifyWeaver's WAM target, then runs the result in the Scittle kernel.

**Run the cells top to bottom.** The generated CLJS (a WAM interpreter, ~3500 lines) is written to files in `/shared/` — not dumped into the notebook.

Requires the updated UnifyWeaver package (Import Package → `unifyweaver_scirepl.zip`) first.


## 1. Load UnifyWeaver

In [ ]:
% Load UnifyWeaver + the WAM-Clojure target.
['../init'].
:- use_module(unifyweaver(targets/wam_clojure_target)).


## 2. Define n-queens (concise Prolog)

In [ ]:
% Concise n-queens (this cell is auto-consulted into the database).
:- dynamic numlist_uw/3, select_uw/3, permutation_uw/2, safe_q/1, safe_q_aux/3, queens_q/2.

numlist_uw(L,H,[L|T]) :- L =< H, L1 is L + 1, numlist_uw(L1,H,T).
numlist_uw(L,H,[])    :- L > H.

select_uw(H,[H|T],T).
select_uw(H,[X|T],[X|T2]) :- select_uw(H,T,T2).

permutation_uw([],[]).
permutation_uw(L,[H|T]) :- select_uw(H,L,Rest), permutation_uw(Rest,T).

safe_q([]).
safe_q([Q|Qs]) :- safe_q_aux(Qs,Q,1), safe_q(Qs).
safe_q_aux([],_,_).
safe_q_aux([Q|Qs],Q0,D0) :- Q =\= Q0+D0, Q =\= Q0-D0, D1 is D0+1, safe_q_aux(Qs,Q0,D1).

queens_q(N,Qs) :- numlist_uw(1,N,L), permutation_uw(L,Qs), safe_q(Qs).


## 3. Transpile → ClojureScript files

In [ ]:
% Transpile the consulted predicates to ClojureScript files in /shared.
once((
    write_wam_clojurescript_files(
        [user:numlist_uw/3, user:select_uw/3, user:permutation_uw/2,
         user:safe_q/1, user:safe_q_aux/3, user:queens_q/2],
        [namespace('generated.nqueens')], '/shared'),
    format("Wrote /shared/runtime.cljs + /shared/core.cljs~n")
)).


## 4. Load the generated ClojureScript

In [ ]:
;; Load the transpiled WAM runtime + predicates from /shared.
(js/scittle.core.eval_string (js/window.sharedVFS.readFile "/shared/runtime.cljs" "utf8"))
(js/scittle.core.eval_string (js/window.sharedVFS.readFile "/shared/core.cljs" "utf8"))
(println "loaded n-queens runtime")


## 5. Run

In [ ]:
;; Check four 4-queens boards. Valid: [2,4,1,3] and [3,1,4,2].
(defn ->wam-list [items]
  (reduce (fn [t x] {:tag :struct :functor "[|]/2" :args [x t]}) "[]" (reverse items)))
(doseq [b [[2 4 1 3] [3 1 4 2] [1 2 3 4] [1 3 2 4]]]
  (println "queens_q(4," b ") =>"
    (generated.nqueens.core/invoke-predicate "queens_q/2" [4 (->wam-list b)])))
